# Demonstrasi: Exploratory Data Analysis (EDA) Terarah — PIF304 Pengantar Data Sains
**Pertemuan 9 — Sub-CPMK S04: Menerapkan eksplorasi data terarah**

EDA terarah **bukan** "membuat semua grafik yang mungkin". EDA terarah berjalan dalam siklus:

**pertanyaan → visual → statistik → catatan (temuan / dugaan / keterbatasan) → pertanyaan berikutnya**

**Dataset:** `Data_Akademik_Sintetis_PDS.xlsx` — **data sintetis (buatan)**, 600 baris, 1 baris = 1 mahasiswa. Tidak ada baris yang berasal dari mahasiswa sungguhan; pola di dalamnya dirancang untuk latihan. Kolom prediktor direkam sampai akhir semester 2; `ipk_sem4`, `status_sem4`, dan `surat_peringatan_sem4` direkam di akhir semester 4.

**Pertanyaan pemandu:** *Faktor tahun pertama apa yang berkaitan dengan IPK semester 4, dan mahasiswa seperti apa yang cenderung berstatus "Perlu pendampingan"?*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

### Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih **`Data_Akademik_Sintetis_PDS.xlsx`** yang sudah Anda unduh dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Silakan pilih file: Data_Akademik_Sintetis_PDS.xlsx")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Google Colab (mis. Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Akademik_Sintetis_PDS.xlsx"

df = pd.read_excel(nama_file)   # sheet pertama = "Data"
print("File:", nama_file, "| ukuran:", df.shape)
df.head()

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})
catatan = []      # buku catatan EDA
def catat(jenis, isi):
    assert jenis in {"temuan", "dugaan", "keterbatasan"}
    catatan.append({"jenis": jenis, "isi": isi})

## 1. Kenali Data Sebelum Bertanya

In [ ]:
profil = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_kosong": df.isna().sum(), "n_unik": df.nunique()})
profil

In [ ]:
df.describe().T.round(2)

### Anomali: nilai yang melanggar data dictionary
Lembar `Data_Dictionary` pada file menyebutkan rentang valid: `kehadiran_pct` 0–100, `jam_belajar_per_minggu` 0–40, `jarak_km` 0–60.

In [ ]:
anomali = {
    "kehadiran_pct > 100":          df["kehadiran_pct"] > 100,
    "jam_belajar_per_minggu > 40":  df["jam_belajar_per_minggu"] > 40,
    "jarak_km > 60":                df["jarak_km"] > 60,
}
for nama, mask in anomali.items():
    print(f"{nama:<30} {int(mask.sum())} baris -> {df.loc[mask, 'id_mhs'].tolist()}")

In [ ]:
# Keputusan EDA: nilai yang MUSTAHIL dijadikan NaN (dicatat); nilai yang MUNGKIN sah (jarak sangat jauh) dibiarkan dan ditandai
eda = df.copy()
eda.loc[eda["kehadiran_pct"] > 100, "kehadiran_pct"] = np.nan
eda.loc[eda["jam_belajar_per_minggu"] > 40, "jam_belajar_per_minggu"] = np.nan
eda["jarak_sangat_jauh"] = eda["jarak_km"] > 60
catat("keterbatasan", "2 nilai kehadiran > 100% dan 1 nilai jam belajar 80 jam dijadikan NaN; 3 nilai jarak > 60 km dibiarkan (mungkin komuter) dan ditandai.")
catat("keterbatasan", f"jam_belajar_per_minggu kosong pada {int(df['jam_belajar_per_minggu'].isna().sum())} baris dan merupakan lapor diri (bisa bias).")
print(eda[["kehadiran_pct", "jam_belajar_per_minggu"]].isna().sum().to_string())

## 2. Lihat Target Lebih Dulu (Univariat)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].hist(eda["ipk_sem4"], bins=np.arange(1.5, 4.05, 0.125), color=TEAL, edgecolor="white")
axes[0].axvline(2.75, color=ORANGE, ls="--"); axes[0].text(2.72, axes[0].get_ylim()[1] * 0.92, "batas 2,75", ha="right", color=NAVY, fontsize=9)
axes[0].set(title="Sebaran IPK semester 4", xlabel="ipk_sem4", ylabel="jumlah mahasiswa")

status = eda["status_sem4"].value_counts()
axes[1].barh(status.index[::-1], status.values[::-1], color=[ORANGE, GREY], height=0.5)
for y, v in enumerate(status.values[::-1]):
    axes[1].text(v + 6, y, f"{v} ({v/len(eda):.0%})", va="center")
axes[1].set(title="29% berstatus 'Perlu pendampingan'", xlabel="jumlah mahasiswa", xlim=(0, 520)); axes[1].grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
catat("temuan", f"IPK sem 4: median {eda['ipk_sem4'].median():.2f} (IQR {eda['ipk_sem4'].quantile(.25):.2f}-{eda['ipk_sem4'].quantile(.75):.2f}); {status['Perlu pendampingan']} dari {len(eda)} ({status['Perlu pendampingan']/len(eda):.0%}) berstatus Perlu pendampingan.")

Dua kelas tidak seimbang (71% : 29%). Ingat angka ini — ia akan menjadi **baseline** di Pertemuan 10.

## 3. Hubungan dengan Variabel Numerik (Bivariat)
*Pertanyaan: variabel numerik tahun pertama mana yang paling erat kaitannya dengan `ipk_sem4`?*

In [ ]:
numerik = ["ips_sem1", "ips_sem2", "kehadiran_pct", "sks_lulus_tahun1", "akses_lms_per_minggu", "jam_belajar_per_minggu", "tugas_terlambat", "jarak_km", "usia_masuk"]
kor = eda[numerik + ["ipk_sem4"]].corr(method="spearman")["ipk_sem4"].drop("ipk_sem4").sort_values()

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(kor.index, kor.values, color=[ORANGE if v < 0 else TEAL for v in kor.values], height=0.6)
for y, v in enumerate(kor.values):
    ax.text(v + (0.02 if v >= 0 else -0.02), y, f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color=NAVY, lw=0.8); ax.set(title="Korelasi Spearman dengan ipk_sem4", xlim=(-0.6, 1.05)); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, kol in zip(axes, ["ips_sem2", "kehadiran_pct", "tugas_terlambat"]):
    sns.regplot(data=eda, x=kol, y="ipk_sem4", ax=ax, scatter_kws={"s": 12, "alpha": 0.35, "color": TEAL}, line_kws={"color": ORANGE, "lw": 2},
                x_jitter=0.15 if kol == "tugas_terlambat" else 0)
    ax.set(title=f"{kol} (rho = {kor[kol]:+.2f})")
plt.tight_layout(); plt.show()
catat("temuan", f"Korelasi Spearman dengan ipk_sem4 paling kuat: ips_sem2 ({kor['ips_sem2']:+.2f}), ips_sem1 ({kor['ips_sem1']:+.2f}), kehadiran_pct ({kor['kehadiran_pct']:+.2f}); tugas_terlambat negatif ({kor['tugas_terlambat']:+.2f}); jarak_km dan usia_masuk mendekati nol.")
catat("dugaan", "Prestasi awal (IPS sem 1-2) dan kebiasaan belajar (kehadiran, ketepatan tugas) adalah kandidat fitur terkuat untuk pemodelan.")

**Catatan kehati-hatian:** `ips_sem1` dan `ips_sem2` sendiri saling berkorelasi kuat. Dua variabel yang hampir kembar membawa informasi yang tumpang-tindih — penting saat memilih fitur (Pertemuan 10).

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.4))
m = eda[numerik + ["ipk_sem4"]].corr(method="spearman")
sns.heatmap(m, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0, linewidths=0.5, annot_kws={"size": 7}, cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Matriks korelasi Spearman (palet divergen, netral di 0)")
plt.tight_layout(); plt.show()

## 4. Hubungan dengan Variabel Kategorikal
*Pertanyaan: apakah IPK semester 4 berbeda menurut jalur masuk, status bekerja, dan beasiswa?*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, kol in zip(axes, ["jalur_masuk", "bekerja_paruh_waktu", "penerima_beasiswa"]):
    sns.boxplot(data=eda, x=kol, y="ipk_sem4", color=TEAL, width=0.5, ax=ax)
    n_per = eda[kol].value_counts()
    ax.set_xticks(range(len(ax.get_xticklabels())), [f"{t.get_text()}\n(n={n_per[t.get_text()]})" for t in ax.get_xticklabels()])
    ax.set(title=kol, xlabel="")
plt.tight_layout(); plt.show()

eda.groupby("penerima_beasiswa")["ipk_sem4"].agg(["count", "median", "mean"]).round(2)

Penerima beasiswa memiliki IPK semester 4 yang jauh lebih tinggi. **Apakah beasiswa meningkatkan IPK?**

Sebelum menyimpulkan, tanyakan: *siapa yang mendapat beasiswa?*

## 5. Segmentasi: Membongkar Hubungan yang Menipu

In [ ]:
print("Rata-rata IPS semester 1 menurut status beasiswa:")
print(eda.groupby("penerima_beasiswa")["ips_sem1"].mean().round(2).to_string(), "\n")

eda["kelompok_ips1"] = pd.cut(eda["ips_sem1"], bins=[0, 2.75, 3.0, 3.25, 3.5, 4.0], labels=["< 2,75", "2,75-3,00", "3,00-3,25", "3,25-3,50", "> 3,50"])
seg = eda.pivot_table(index="kelompok_ips1", columns="penerima_beasiswa", values="ipk_sem4", aggfunc=["mean", "count"], observed=True).round(2)
seg

In [ ]:
selisih_kasar = eda.groupby("penerima_beasiswa")["ipk_sem4"].mean().diff().iloc[-1]
rata_seg = eda.pivot_table(index="kelompok_ips1", columns="penerima_beasiswa", values="ipk_sem4", aggfunc="mean", observed=True)
n_seg = eda.pivot_table(index="kelompok_ips1", columns="penerima_beasiswa", values="ipk_sem4", aggfunc="count", observed=True)
cukup = n_seg["Ya"] >= 10                      # kelompok dengan penerima beasiswa < 10 orang tidak ditafsirkan
selisih_seg = (rata_seg["Ya"] - rata_seg["Tidak"])[cukup]
selisih_setara = float((selisih_seg * n_seg.loc[cukup, "Ya"]).sum() / n_seg.loc[cukup, "Ya"].sum())   # rata-rata tertimbang
print(f"Selisih rata-rata IPK (Ya - Tidak), semua mahasiswa      : {selisih_kasar:+.2f}")
print("Selisih di dalam kelompok IPS sem 1 yang setara (n Ya >= 10):")
print(selisih_seg.round(2).to_string())
print(f"Rata-rata tertimbang selisih di dalam kelompok          : {selisih_setara:+.2f}")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
tampil = rata_seg[cukup]
x = np.arange(len(tampil)); lebar = 0.36
for i, (k, w) in enumerate(zip(["Tidak", "Ya"], PALET[:2])):
    ax.bar(x + (i - 0.5) * (lebar + 0.03), tampil[k], width=lebar, color=w, label=f"Beasiswa: {k}")
ax.set_xticks(x, tampil.index); ax.set(title="Pada IPS sem 1 yang setara, IPK sem 4 hampir sama", xlabel="kelompok ips_sem1", ylabel="rata-rata ipk_sem4", ylim=(0, 4.4))
ax.legend(frameon=False, ncol=2, loc="upper left"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

catat("temuan", f"Penerima beasiswa rata-rata ipk_sem4 lebih tinggi {selisih_kasar:+.2f}; di dalam kelompok IPS sem 1 yang setara selisihnya rata-rata tinggal {selisih_setara:+.2f}.")
catat("dugaan", "Selisih IPK penerima beasiswa terutama mencerminkan seleksi (beasiswa diberikan kepada yang IPS-nya sudah tinggi), bukan efek beasiswa.")
catat("keterbatasan", "Data observasional: tidak dapat memisahkan efek beasiswa dari seleksi penerimanya.")

**Pelajaran:** selisih +0,40 pada perbandingan kasar menyusut menjadi kurang dari +0,10 setelah mahasiswa dibandingkan dengan yang **setara** prestasi awalnya. (Kelompok IPS < 2,75 hanya berisi 2 penerima beasiswa, sehingga tidak ditafsirkan.) Beasiswa diberikan kepada mahasiswa yang IPS-nya sudah tinggi (*seleksi*), sehingga `ips_sem1` menjadi **variabel pengganggu** (*confounder*).

Segmentasi tidak membuktikan sebab-akibat, tetapi sering cukup untuk **mencegah kesimpulan yang keliru**.

## 6. Profil Kelompok "Perlu Pendampingan"

In [ ]:
banding = eda.groupby("status_sem4")[["ips_sem1", "ips_sem2", "kehadiran_pct", "jam_belajar_per_minggu", "akses_lms_per_minggu", "tugas_terlambat", "sks_lulus_tahun1"]].median().T
banding["selisih"] = banding["Perlu pendampingan"] - banding["Aman"]
banding.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, kol in zip(axes, ["kehadiran_pct", "ips_sem2"]):
    sns.kdeplot(data=eda, x=kol, hue="status_sem4", hue_order=["Aman", "Perlu pendampingan"], palette=[TEAL, ORANGE], common_norm=False, fill=True, alpha=0.25, ax=ax)
    ax.set(title=f"Sebaran {kol} menurut status", ylabel="kepadatan")
plt.tight_layout(); plt.show()
catat("temuan", "Kelompok Perlu pendampingan memiliki median IPS sem 2, kehadiran, dan akses LMS lebih rendah serta tugas terlambat lebih banyak; sebaran kedua kelompok tetap tumpang-tindih.")
catat("dugaan", "Aturan sederhana berbasis ips_sem2 mungkin sudah cukup baik sebagai baseline; model perlu mengunggulinya.")

## 7. Kolom yang "Terlalu Bagus"

In [ ]:
pd.crosstab(eda["surat_peringatan_sem4"], eda["status_sem4"])

Semua penerima surat peringatan berstatus "Perlu pendampingan". Hubungan yang nyaris sempurna seperti ini harus memicu pertanyaan: **kapan kolom ini direkam?** Menurut data dictionary: akhir semester 4 — bersamaan dengan target. Kolom ini adalah **akibat** dari target, bukan prediktor. Kita bahas tuntas di Pertemuan 10 (*data leakage*).

In [ ]:
catat("keterbatasan", "surat_peringatan_sem4 direkam bersamaan dengan target (akhir sem 4) sehingga tidak boleh dipakai sebagai prediktor.")
catat("keterbatasan", "Data sintetis: pola hanya untuk latihan, tidak menggambarkan mahasiswa nyata.")

## 8. Ringkasan Temuan EDA

In [ ]:
ringkasan = pd.DataFrame(catatan)
for jenis in ["temuan", "dugaan", "keterbatasan"]:
    print(f"\n=== {jenis.upper()} ===")
    for i, isi in enumerate(ringkasan.loc[ringkasan["jenis"] == jenis, "isi"], 1):
        print(f"{i}. {isi}")
ringkasan.to_csv("ringkasan_eda_demo_P9.csv", index=False)

### Tiga jenis pernyataan — jangan dicampur
| Jenis | Ciri | Contoh kalimat |
|---|---|---|
| **Temuan** | Terlihat langsung di data; ada angkanya | "Korelasi kehadiran dengan IPK sem 4 adalah +0,6" |
| **Dugaan** | Penjelasan/hipotesis; perlu diuji | "Kehadiran mungkin mencerminkan disiplin belajar" |
| **Keterbatasan** | Hal yang membatasi klaim | "Jam belajar adalah lapor diri dan 4% kosong" |

**Berikutnya (Pertemuan 10):** memilih fitur, membagi data, mencegah kebocoran, dan membangun baseline.